In [1]:
#lib 
from langchain.llms import HuggingFacePipeline
from langchain_core.runnables import Runnable
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
from sentence_transformers import SentenceTransformer, util
from faster_whisper import WhisperModel
import sounddevice as sd
import soundfile as sf
import torch
import numpy as np
import warnings
import os 

warnings.filterwarnings('ignore')

In [2]:
#tts Question ready {use different model this one is not good}

tts_model_name = "facebook/mms-tts-eng"
tts_process = AutoProcessor.from_pretrained(tts_model_name)
tts_model = AutoModel.from_pretrained(tts_model_name)

question = "what is AI"
inputs = tts_process(question, return_tensors='pt')

with torch.no_grad():
    speech = tts_model(**inputs).waveform
    
audio = speech[0].numpy().astype(np.float32)
sd.play(audio, samplerate=16000, blocking=True)

Exception ignored from cffi callback <function _StreamBase.__init__.<locals>.finished_callback_wrapper at 0x00000194BADBAB60>:
Traceback (most recent call last):
  File "C:\Users\Sumit\anaconda3\Lib\site-packages\sounddevice.py", line 940, in finished_callback_wrapper
    return finished_callback()
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Sumit\anaconda3\Lib\site-packages\sounddevice.py", line 2653, in finished_callback
    del self.out
        ^^^^^^^^
AttributeError: '_CallbackContext' object has no attribute 'out'


In [3]:
#question stt question

class WhisperTranscribe(Runnable):
    def __init__(self, model_size="small"):
        self.model =WhisperModel(
                        model_size_or_path="small",
                        device="cpu",
                        compute_type="int8")
        
    def invoke(self, audio_path: str):
        segments, info = self.model.transcribe(
            audio_path, vad_filter=True, beam_size=5,temperature=0.0, language="en", #none for 
        )
        
        
        text = " ".join(segment.text for segment in segments)
        return {
            "text":text.strip(), 
            "language": info.language,
            "duration": info.duration
        }
    
whisper = WhisperTranscribe()
output = whisper.invoke("audio1.wav")
print(output)

{'text': 'Ixophical intelligence refers to a range of cognitive abilities that are developed during  early childhood and function in everyday life.', 'language': 'en', 'duration': 8.688}


In [4]:
#llm text - generation {use tiny lama model not this }

llm_model_name = "Qwen/Qwen1.5-0.5B-Chat"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_model_name)
llm_pipe = pipeline('text-generation',model=llm_model, tokenizer=llm_tokenizer, torch_dtype='int8')
llm = HuggingFacePipeline(pipeline=llm_pipe)

gen_answer = llm.invoke(question)
print(gen_answer)

Device set to use cpu


what is AI?
AI stands for artificial intelligence. It refers to the ability of computers and machines to perform tasks that


In [7]:
#Compare llm and audio answer with score 
user_answer = "AI stands for Artificial Intelligence. It is the simulation of human intelligence in machines that can perform tasks"


similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
user_answer = similarity_model.encode(output['text'])
llm_answer = similarity_model.encode(gen_answer)

Similarity = util.cos_sim(user_answer,llm_answer)
print("Score: ", Similarity.item())

Score:  0.4410015046596527


In [13]:
# tts report
